# Basic Data Cleaning and Analysis

This notebook covers: `out/math_results.csv`, `out/quiz_results.csv`, `out/gymitrainer.csv`, `out/events/g.csv`, `out/events/m.csv`, and `out/events/s.csv`.

In [1]:
!pip install numpy
!pip install pandas
!pip install matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from pathlib import Path
import ast
import json
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)


In [3]:
BASE_DIR = Path('out')
EVENTS_DIR = BASE_DIR / 'events'

DATASET_PATHS = {
    'math_results': BASE_DIR / 'math_results.csv',
    'quiz_results': BASE_DIR / 'quiz_results.csv',
    'gymitrainer': BASE_DIR / 'gymitrainer.csv',
    'events_g': EVENTS_DIR / 'g.csv',
    'events_m': EVENTS_DIR / 'm.csv',
    'events_s': EVENTS_DIR / 's.csv',
}

def load_csv(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing dataset: {path}')
    return pd.read_csv(path, low_memory=False, **kwargs)

def basic_profile(df: pd.DataFrame, name: str) -> None:
    print('=' * 90)
    print(f'{name}')
    print('=' * 90)
    print(f'shape: {df.shape}')
    print(f'duplicates: {int(df.duplicated().sum())}')
    print('missing values:')
    print(df.isna().sum().sort_values(ascending=False).head(10))
    print('\ndtypes:')
    print(df.dtypes)
    print('\nhead:')
    print(df.head(3))
    print()

def drop_unnamed_columns(df: pd.DataFrame) -> pd.DataFrame:
    unnamed = [c for c in df.columns if str(c).lower().startswith('unnamed')]
    return df.drop(columns=unnamed, errors='ignore')

def coerce_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

def parse_unix_time(df: pd.DataFrame, col: str, unit: str, out_col: str | None = None) -> pd.DataFrame:
    if col in df.columns:
        target = out_col or f'{col}_dt'
        df[target] = pd.to_datetime(df[col], unit=unit, errors='coerce', utc=True)
    return df

def safe_parse_list(text):
    if pd.isna(text):
        return None
    if isinstance(text, list):
        return text
    text = str(text).strip()
    if text == '' or text == '[]':
        return []
    try:
        return ast.literal_eval(text)
    except Exception:
        return None


In [4]:
raw = {name: load_csv(path) for name, path in DATASET_PATHS.items()}
for name, df in raw.items():
    basic_profile(df, f'RAW -> {name}')


RAW -> math_results
shape: (12060, 9)
duplicates: 0
missing values:
answer           5061
Unnamed: 0          0
session_id          0
question_part       0
user_id             0
question_id         0
points              0
max_points          0
timestamp           0
dtype: int64

dtypes:
Unnamed: 0         int64
session_id           str
question_part      int64
user_id            int64
question_id          str
points           float64
max_points       float64
answer               str
timestamp          int64
dtype: object

head:
   Unnamed: 0                session_id  question_part  user_id               question_id  points  max_points answer   timestamp
0           0  679a2fdce127578405afeef5              0      490  66a5eb558948dda8b2fd63e3     0.0         4.0    NaN  1738158044
1           1  679a2fdce127578405afeef5              0      490  66a5eb558948dda8b2fd63e1     0.0         2.0    NaN  1738158044
2           2  679a2fdce127578405afeef5              1      490  66a5eb558948dd

In [5]:
cleaned = {}

# math_results
math_df = raw['math_results'].copy()
math_df = drop_unnamed_columns(math_df)
math_df = coerce_numeric(math_df, ['question_part', 'user_id', 'points', 'max_points', 'timestamp'])
math_df = parse_unix_time(math_df, 'timestamp', unit='s', out_col='timestamp_dt')
math_df = math_df.drop(['answer'], axis=1)
math_df = math_df.drop_duplicates().reset_index(drop=True)
cleaned['math_results'] = math_df

# quiz_results
quiz_df = raw['quiz_results'].copy()
quiz_df = drop_unnamed_columns(quiz_df)
quiz_numeric = ['session_id', 'user_id', 'course_id', 'quiz_id', 'question_id', 'points', 'max_points', 'time', 'hint_count', 'time_spent', 'correct']
quiz_df = coerce_numeric(quiz_df, quiz_numeric)
quiz_df = parse_unix_time(quiz_df, 'time', unit='s', out_col='time_dt')
quiz_df = quiz_df.drop_duplicates().reset_index(drop=True)
cleaned['quiz_results'] = quiz_df

# gymitrainer
gym_df = raw['gymitrainer'].copy()
gym_df = drop_unnamed_columns(gym_df)
if 'startTime' in gym_df.columns:
    gym_df['startTime'] = pd.to_numeric(gym_df['startTime'], errors='coerce')
    gym_df = parse_unix_time(gym_df, 'startTime', unit='s', out_col='startTime_dt')
if 'content' in gym_df.columns:
    gym_df['content_parsed'] = gym_df['content'].apply(safe_parse_list)
    gym_df['message_count'] = gym_df['content_parsed'].apply(lambda x: len(x) if isinstance(x, list) else np.nan)
    gym_df = gym_df.drop(columns=['content_parsed'])
gym_df = gym_df.drop_duplicates().reset_index(drop=True)
cleaned['gymitrainer'] = gym_df

# events (g, m, s)
for event_name in ['events_g', 'events_m', 'events_s']:
    ev_df = raw[event_name].copy()
    ev_df = drop_unnamed_columns(ev_df)

    # Most event files use timestamp in milliseconds.
    if 'timestamp' in ev_df.columns:
        ev_df['timestamp'] = pd.to_numeric(ev_df['timestamp'], errors='coerce')
        ev_df = parse_unix_time(ev_df, 'timestamp', unit='ms', out_col='timestamp_dt')

    numeric_guess = ['event', 'pageview_id', 'currentTime', 'duration', 'mediaId', 'windowHeight', 'fullHeight']
    ev_df = coerce_numeric(ev_df, numeric_guess)

    # Remove exact duplicate rows common in event logs.
    ev_df = ev_df.drop_duplicates().reset_index(drop=True)
    cleaned[event_name] = ev_df

# Specific cleaning for events_g
print(f"\nDropping {cleaned['events_g']['event'].isna().sum()} NaN rows in events_g 'event' column.")
cleaned['events_g'] = cleaned['events_g'].dropna(subset=['event'])
cleaned['events_g']['event'] = cleaned['events_g']['event'].astype(int)

# Specific cleaning for events_m
print("\nDropping url column from events_m due to excessive missing values. Sample of non-missing urls:")
print(raw['events_m'][~raw['events_m']['url'].isna()]['url'].to_string())
cleaned['events_m'] = cleaned['events_m'].drop(['url'], axis=1)

# Specific cleaning for events_s
print(f"\nDropping {cleaned['events_s']['fullHeight'].isna().sum()} NaN rows in events_s 'fullHeight' column.")
cleaned['events_s'] = cleaned['events_s'].dropna(subset=['fullHeight'])

print('\nCleaning completed for datasets:', ', '.join(cleaned.keys()))



Dropping 1 NaN rows in events_g 'event' column.

Dropping url column from events_m due to excessive missing values. Sample of non-missing urls:
516833    HIfIpLay8BM
516834     1104054768
516835     1104054768
516836     1104054768
516837     1104054768
516838     1104054768

Dropping 102 NaN rows in events_s 'fullHeight' column.

Cleaning completed for datasets: math_results, quiz_results, gymitrainer, events_g, events_m, events_s


In [6]:
for name, df in cleaned.items():
    basic_profile(df, f'CLEANED -> {name}')

CLEANED -> math_results
shape: (12060, 8)
duplicates: 0
missing values:
session_id       0
question_part    0
user_id          0
question_id      0
points           0
max_points       0
timestamp        0
timestamp_dt     0
dtype: int64

dtypes:
session_id                      str
question_part                 int64
user_id                       int64
question_id                     str
points                      float64
max_points                  float64
timestamp                     int64
timestamp_dt     datetime64[s, UTC]
dtype: object

head:
                 session_id  question_part  user_id               question_id  points  max_points   timestamp              timestamp_dt
0  679a2fdce127578405afeef5              0      490  66a5eb558948dda8b2fd63e3     0.0         4.0  1738158044 2025-01-29 13:40:44+00:00
1  679a2fdce127578405afeef5              0      490  66a5eb558948dda8b2fd63e1     0.0         2.0  1738158044 2025-01-29 13:40:44+00:00
2  679a2fdce127578405afeef5          

#### **quiz_results.csv: Group to {user_id, quiz_id, min(timestamp)}**

In [7]:
cleaned['quiz_results'].head(2)

,session_id,user_id,course_id,quiz_id,question_id,points,max_points,time,hint_count,time_spent,correct,time_dt
0,1,2437,42,2,1,0,1,1665067915,0,0,0,2022-10-06 14:51:55+00:00
1,2,2437,42,2,2,0,1,1665069310,0,0,0,2022-10-06 15:15:10+00:00


In [8]:
grouped_quiz_results = cleaned['quiz_results'].groupby(["user_id", "quiz_id"], as_index=False).agg(
            time_quiz_was_given=("time_dt", "min"),
            quiz_sum_points=("points", "sum"),
            quiz_sum_max_points=("max_points", "sum"),
            sum_hint_count=("hint_count", "sum"),
            mean_time_spent=("time_spent", "mean"),
        ).sort_values(["user_id", "quiz_id"])

In [9]:
grouped_quiz_results.shape

(21449, 7)

In [10]:
grouped_quiz_results.head()

,user_id,quiz_id,time_quiz_was_given,quiz_sum_points,quiz_sum_max_points,sum_hint_count,mean_time_spent
0,9,40,2025-01-22 14:52:09+00:00,1,5,0,111.714286
1,9,41,2025-01-22 14:34:53+00:00,3,5,0,43.000000
2,9,46,2024-12-29 17:25:48+00:00,4,8,3,239.000000
3,9,51,2024-12-11 16:09:53+00:00,2,9,1,56.000000
4,9,53,2024-12-11 16:43:17+00:00,8,8,1,168.625000


#### **math_results.csv: Group to {user_id, timestamp}**

In [11]:
cleaned['math_results'].head(2)

,session_id,question_part,user_id,question_id,points,max_points,timestamp,timestamp_dt
0,679a2fdce127578405afeef5,0,490,66a5eb558948dda8b2fd63e3,0.0,4.0,1738158044,2025-01-29 13:40:44+00:00
1,679a2fdce127578405afeef5,0,490,66a5eb558948dda8b2fd63e1,0.0,2.0,1738158044,2025-01-29 13:40:44+00:00


In [12]:
grouped_maths_df = math_df.groupby(["user_id", "timestamp_dt"], as_index=False).agg(
               math_sum_points=("points", "sum"),
               math_sum_max_points=("max_points", "sum"),
               number_of_questions_solved=("user_id", "size"),
           ).sort_values(["user_id", "timestamp_dt"])

In [13]:
grouped_maths_df.shape

(591, 5)

In [14]:
grouped_maths_df.head()

,user_id,timestamp_dt,math_sum_points,math_sum_max_points,number_of_questions_solved
0,20,2025-02-21 08:43:03+00:00,4.0,36.00000,10
1,33,2025-01-23 10:45:55+00:00,9.0,40.00002,28
2,52,2025-02-16 17:47:27+00:00,2.0,32.00000,21
3,52,2025-02-19 16:50:21+00:00,4.0,34.00000,15
4,95,2025-01-23 19:24:18+00:00,13.5,36.00000,18


#### Combine the grouped_quiz_results with grouped_maths_df such that for each row of grouped_quiz_results df, we have an aggregation of the rows of grouped_maths_df whose **'timestamp_dt' < 'time_quiz_was_given'** of the grouped_quiz_results row

In [15]:
# Merge on user_id to get all combinations, then filter by timestamp condition
merged = grouped_quiz_results.merge(
    grouped_maths_df[["user_id", "timestamp_dt", "math_sum_points", "math_sum_max_points", "number_of_questions_solved"]],
    on="user_id",
    how="left"
)

print('\nBefore filtering by timestamp condition:')
print(merged[merged.user_id == 7154].to_string())

# Keep only rows where maths timestamp is BEFORE the quiz was given
merged = merged[merged["timestamp_dt"] < merged["time_quiz_was_given"]]

print('\nAfter filtering by timestamp condition:')
print(merged[merged.user_id == 7154].to_string())

# Aggregate the maths rows per (user_id, quiz_id)
maths_prior_agg = merged.groupby(["user_id", "quiz_id"], as_index=False).agg(
    maths_sum_points=("math_sum_points", "sum"),
    maths_sum_max_points=("math_sum_max_points", "sum"),
    maths_total_questions_solved=("number_of_questions_solved", "sum"),
    maths_sessions_before_quiz=("timestamp_dt", "count"),
)

print('\nAggregated maths')
print(maths_prior_agg[maths_prior_agg.user_id == 7154].to_string())

# Join the aggregation back to grouped_quiz_results
result = grouped_quiz_results.merge(
    maths_prior_agg,
    on=["user_id", "quiz_id"],
    how="inner"
)

print("\nFinal result after joining aggregated maths with quiz results:")
print(result[result.user_id == 7154].to_string())


Before filtering by timestamp condition:
       user_id  quiz_id       time_quiz_was_given  quiz_sum_points  quiz_sum_max_points  sum_hint_count  mean_time_spent              timestamp_dt  math_sum_points  math_sum_max_points  number_of_questions_solved
27668     7154       12 2026-02-28 09:16:42+00:00                3                   14               0         1.000000 2026-02-27 16:37:03+00:00           12.332               35.998                        21.0
27669     7154       12 2026-02-28 09:16:42+00:00                3                   14               0         1.000000 2026-02-27 18:33:27+00:00            0.000               36.000                        16.0
27670     7154       12 2026-02-28 09:16:42+00:00                3                   14               0         1.000000 2026-02-28 09:05:59+00:00            2.000               36.000                        16.0
27671     7154       17 2025-11-19 14:50:41+00:00                5                   10               0   

In [18]:
result.user_id.nunique()

101

In [17]:
result.head()

,user_id,quiz_id,time_quiz_was_given,quiz_sum_points,quiz_sum_max_points,sum_hint_count,mean_time_spent,maths_sum_points,maths_sum_max_points,maths_total_questions_solved,maths_sessions_before_quiz
0,114,65,2025-02-09 13:41:32+00:00,7,10,0,21.100000,27.0,108.00000,54.0,3
1,282,40,2025-07-30 20:44:56+00:00,4,25,0,621.514286,4.0,72.00007,35.0,2
2,559,42,2025-11-24 15:21:46+00:00,3,8,0,11.750000,0.0,32.00000,21.0,1
3,559,48,2025-11-24 12:01:28+00:00,2,8,0,9.875000,0.0,32.00000,21.0,1
4,564,30,2025-09-26 07:48:37+00:00,1,7,0,17.142857,0.0,33.00400,30.0,1
